In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

retail_df = pd.read_csv('retail_prices.csv', parse_dates=['date'])
cpi_df = pd.read_csv('cpi.csv', parse_dates=['date'])

display(retail_df.head(2))
display(cpi_df.head(2))

,date,food_type,sub_category,commodity,price
0,2020-01-01,Calorie-Dense,Cereals,"RICE, REGULAR-MILLED, 1 KG",35.80
1,2020-02-01,Calorie-Dense,Cereals,"RICE, REGULAR-MILLED, 1 KG",35.92


,date,income_group,food_type,sub_category,cpi
0,2020-01-01,All Income,Baseline,ALL ITEMS,103.9
1,2020-02-01,All Income,Baseline,ALL ITEMS,103.4


In [14]:
# calculate inflation
rq1 = retail_df.groupby(['date', 'food_type'])['price'].mean().reset_index()
rq1['inf'] = rq1.groupby('food_type')['price'].pct_change(12) * 100
rq1 = rq1.dropna(subset=['inf'])

cal = rq1[rq1['food_type'] == 'Calorie-Dense']['inf']
nut = rq1[rq1['food_type'] == 'Nutrient-Dense']['inf']

# distribution checks
print(f"Shapiro Cal: {stats.shapiro(cal)[1]:.4f}")
print(f"Shapiro Nut: {stats.shapiro(nut)[1]:.4f}")

# 0.0839 and 0.0674 are both above 0.05, so we can assume normality for both distributions

# jump straight to t-test
t_stat, p_val = stats.ttest_ind(nut, cal, equal_var=False)
print(f"T-test stat: {t_stat:.4f} | p: {p_val:.4f}")

Shapiro Cal: 0.0839
Shapiro Nut: 0.0674
T-test stat: -1.2732 | p: 0.2052


In [ ]:
# get baseline inflation
base = cpi_df[cpi_df['sub_category'] == 'ALL ITEMS'].copy()
base['inf'] = base.groupby('income_group')['cpi'].pct_change(12) * 100
rq2 = base.dropna(subset=['inf'])

b30 = rq2[rq2['income_group'] == 'Bottom 30%']['inf'].values
all_inc = rq2[rq2['income_group'] == 'All Income']['inf'].values

# distribution checks
print(f"Shapiro B30: {stats.shapiro(b30)[1]:.4e}")
print(f"Shapiro All: {stats.shapiro(all_inc)[1]:.4e}")

# 2.6891e-04 and 2.9620e-05 are both below 0.05, so we reject the null hypothesis of normality for both distributions

# jump straight to wilcoxon
stat, p_val = stats.wilcoxon(b30, all_inc, alternative='greater')
print(f"Wilcoxon stat: {stat:.4f} | p: {p_val:.4e}")

Shapiro B30: 2.6891e-04
Shapiro All: 2.9620e-05
Wilcoxon stat: 1888.0000 | p: 6.2436e-06
